In [1]:
import pandas as pd

df = pd.read_csv("noised_unanchoredevents.csv", sep=";")

df.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
s = df["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 962480
European dd.mm.yyyy 239373
European d.mm.yyyy 239373
US-like mm/dd/yyyy 0


/var/folders/sn/nh99ycnd55q_9cvc_vgczkfw0000gn/T/ipykernel_92293/2862589946.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


invalid / custom strings 0
UTC 193657


In [3]:
def clean_unanchored_events_mixed_formats(
    df,
    timestamp_column="time:timestamp"
):
    df = df.copy()

    timestamps = df[timestamp_column].astype("string")

    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

    # Format 1: German format, e.g. 31.12.2017 14:30:00
    mask_german = timestamps.str.match(
        r"^\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}:\d{2}$",
        na=False
    )

    parsed.loc[mask_german] = pd.to_datetime(
        timestamps.loc[mask_german],
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce",
        utc=True
    )

    # Format 2: ISO with T and Z, e.g. 2017-12-31T14:30:00.123456Z
    mask_iso_z = timestamps.str.match(
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$",
        na=False
    )

    parsed.loc[mask_iso_z] = pd.to_datetime(
        timestamps.loc[mask_iso_z],
        format="%Y-%m-%dT%H:%M:%S.%fZ",
        errors="coerce",
        utc=True
    )

    # Fallback: try normal pandas parsing for unchanged/default timestamps
    mask_remaining = parsed.isna() & timestamps.notna()

    parsed.loc[mask_remaining] = pd.to_datetime(
        timestamps.loc[mask_remaining],
        errors="coerce",
        utc=True
    )

    df[timestamp_column] = parsed

    return df

In [4]:
df_cleaned = clean_unanchored_events_mixed_formats(
    df,
    timestamp_column="time:timestamp"
)

In [5]:
df_cleaned.head(100)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Obtained,User_116,W_Validate application,Workflow,Workitem_1121078170,start,2016-01-13 08:57:21.342000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,statechange,User_116,A_Validating,Application,ApplState_145257449,complete,2016-01-13 08:57:22+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,statechange,User_116,O_Returned,Offer,OfferState_836197344,complete,2016-01-13 09:08:14.217000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_997411923
98,Released,User_116,W_Validate application,Workflow,Workitem_1394551701,suspend,2016-01-13 09:14:02.550000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
s = df_cleaned["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 1201090
European dd.mm.yyyy 0
European d.mm.yyyy 0
US-like mm/dd/yyyy 0


/var/folders/sn/nh99ycnd55q_9cvc_vgczkfw0000gn/T/ipykernel_92293/4011362491.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


invalid / custom strings 0
UTC 0


In [8]:
len(df_cleaned)

1202267

In [9]:
df_check = df_cleaned.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    222
Offer          192
Workflow       763
Name: time:timestamp, dtype: int64


In [10]:
df_check2 = df.copy()

print(
    df_check2.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    222
Offer          192
Workflow         0
Name: time:timestamp, dtype: int64


In [11]:
df["EventOrigin"].value_counts()

EventOrigin
Workflow       768823
Application    239595
Offer          193849
Name: count, dtype: int64

In [12]:
application_na_rows = df_check[
    (df_check["EventOrigin"] == "Application") &
    (df_check["time:timestamp"].isna())
]

In [22]:
application_na_rows[
    ["case:concept:name", "EventID", "case:concept:name", "time:timestamp", "concept:name"]
].head(100)

,case:concept:name,EventID,case:concept:name,time:timestamp,concept:name
16251,Application_1462703151,ApplState_1428953723,Application_1462703151,NaT,A_Validating
31740,Application_953097922,ApplState_1056054725,Application_953097922,NaT,A_Complete
48451,Application_1979616665,ApplState_1591021281,Application_1979616665,NaT,A_Complete
67302,Application_2122367472,ApplState_1153503683,Application_2122367472,NaT,A_Accepted
70429,Application_1901466720,ApplState_768542349,Application_1901466720,NaT,A_Complete
...,...,...,...,...,...
469269,Application_1548417050,Application_1548417050,Application_1548417050,NaT,A_Create Application
470788,Application_1702985007,Application_1702985007,Application_1702985007,NaT,A_Create Application
472476,Application_290904207,Application_290904207,Application_290904207,NaT,A_Create Application
475829,Application_1053355914,ApplState_1238406429,Application_1053355914,NaT,A_Validating


In [25]:
case_id = application_na_rows.iloc[4]["case:concept:name"]

df_check.loc[
    df_check["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "EventID"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,EventID
70413,Application_1901466720,A_Create Application,Application,2016-01-25 19:23:50+00:00,Application_1901466720
70414,Application_1901466720,A_Submitted,Application,2016-01-25 19:23:50+00:00,ApplState_995886280
70415,Application_1901466720,W_Handle leads,Workflow,2016-01-25 19:23:50.693000+00:00,Workitem_1678848309
70416,Application_1901466720,W_Handle leads,Workflow,2016-01-25 19:25:13.595000+00:00,Workitem_844655806
70417,Application_1901466720,W_Complete application,Workflow,2016-01-25 19:25:13.614000+00:00,Workitem_528844293
70418,Application_1901466720,A_Concept,Application,2016-01-25 19:25:13+00:00,ApplState_798863895
70419,Application_1901466720,W_Complete application,Workflow,2016-01-26 08:28:59.984000+00:00,Workitem_1755039725
70420,Application_1901466720,W_Complete application,Workflow,2016-01-26 08:30:03.726000+00:00,Workitem_509194137
70421,Application_1901466720,W_Complete application,Workflow,2016-01-26 08:30:55.523000+00:00,Workitem_1636133697
70422,Application_1901466720,A_Accepted,Application,2016-01-26 08:47:00+00:00,ApplState_1800618350


In [23]:
df_check["event_order"] = range(len(df_check))

In [24]:
def impute_missing_timestamps_within_cases(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()
    imputable_mask = missing_mask & previous_time.notna() & next_time.notna()

    df.loc[imputable_mask, timestamp_column] = (
        previous_time[imputable_mask]
        + (next_time[imputable_mask] - previous_time[imputable_mask]) / 2
    )

    df.loc[imputable_mask, "timestamp_cleaning"] = "imputed_between_neighbors"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [26]:
df_cleaned = impute_missing_timestamps_within_cases(
    df_check,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column="event_order"
)

In [27]:
df_cleaned.loc[
    df_cleaned["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "timestamp_cleaning"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,timestamp_cleaning
70413,Application_1901466720,A_Create Application,Application,2016-01-25 19:23:50+00:00,NaN
70414,Application_1901466720,A_Submitted,Application,2016-01-25 19:23:50+00:00,NaN
70415,Application_1901466720,W_Handle leads,Workflow,2016-01-25 19:23:50.693000+00:00,NaN
70416,Application_1901466720,W_Handle leads,Workflow,2016-01-25 19:25:13.595000+00:00,NaN
70417,Application_1901466720,W_Complete application,Workflow,2016-01-25 19:25:13.614000+00:00,NaN
70418,Application_1901466720,A_Concept,Application,2016-01-25 19:25:13+00:00,NaN
70419,Application_1901466720,W_Complete application,Workflow,2016-01-26 08:28:59.984000+00:00,NaN
70420,Application_1901466720,W_Complete application,Workflow,2016-01-26 08:30:03.726000+00:00,NaN
70421,Application_1901466720,W_Complete application,Workflow,2016-01-26 08:30:55.523000+00:00,NaN
70422,Application_1901466720,A_Accepted,Application,2016-01-26 08:47:00+00:00,NaN


In [28]:
df_cleaned["timestamp_cleaning"].value_counts()

timestamp_cleaning
imputed_between_neighbors    1115
Name: count, dtype: int64

In [29]:
def fill_missing_timestamps_with_neighbor(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None,
    cleaning_column="timestamp_cleaning"
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()

    previous_mask = missing_mask & previous_time.notna()
    df.loc[previous_mask, timestamp_column] = previous_time[previous_mask]
    df.loc[previous_mask, cleaning_column] = "filled_from_previous_timestamp"

    missing_mask = df[timestamp_column].isna()

    next_mask = missing_mask & next_time.notna()
    df.loc[next_mask, timestamp_column] = next_time[next_mask]
    df.loc[next_mask, cleaning_column] = "filled_from_next_timestamp"

    missing_mask = df[timestamp_column].isna()
    df.loc[missing_mask, cleaning_column] = "could_not_fill_timestamp"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [30]:
df_cleaned = fill_missing_timestamps_with_neighbor(
    df_cleaned,
    case_column="case:concept:name",
    timestamp_column="time:timestamp"
)

In [31]:
df_cleaned["timestamp_cleaning"].value_counts()

timestamp_cleaning
imputed_between_neighbors         1115
filled_from_previous_timestamp      34
filled_from_next_timestamp          28
Name: count, dtype: int64